# Lesson 17 - Practice: Testing with pytest

Work in `lessons/lesson-17-testing/` (point a terminal at that folder).

Before you start, make sure pytest is installed in the kernel's Python:

```
python -m pytest --version
```

How to work through this notebook:

1. Read the **Task**, **File tree**, **Interface contract** and **Constraints**.
2. Create each file with a source editor (VS Code/Jupyter editor).
3. From the terminal, run the test command shown in **Constraints**.
4. Run the **Driver** cell - it runs the same tests and pins the expected summary.
5. Compare the Driver output with **Expected** before moving on.

The Driver cells print only the stable summary line (`N passed in 0.01s`). In your
terminal you will also see progress dots and real timing - cosmetics, ignore them.

E1 -> M3 build on each other; do them in order. A wrong Driver output usually means
a file from an earlier exercise is off - fix it first, then continue.

## E1 - First tests: assert, discovery, running pytest

**Type:** REAL-WORLD - easy - writes 2 files

**Task**
A shop computes sale prices. `apply_discount(100.0, 10)` must mean "10 percent
off 100.00" -> `90.0`. Write the function, then write tests that prove it.

**File tree** (create in the lesson folder):

```
discounter.py
test_discounter.py
```

**Interface contract**
`apply_discount(price: float, percent: int) -> float` - returns `price` reduced
by `percent` percent, rounded to 2 decimal places.

**Constraints**
- `discounter.py` holds one function and nothing else.
- `test_discounter.py` holds two test functions; each asserts exactly one behavior.
- Command: `python -m pytest test_discounter.py -q`

**Expected** (Driver cell): `2 passed in 0.01s`

In [4]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_discounter.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

2 passed in 0.01s


## E2 - Read a failure report, then fix the TEST (not the code)

**Type:** REAL-WORLD - easy - writes 2 files

**Task**
`messenger.py` greets a person and returns `"hello, Ada"` (lowercase h). Write a
test that **expects the wrong thing on purpose** (`"Hello, Ada"`), run it, and
read the failure report. Then fix the **test** to assert what the function really
returns. Do not touch `messenger.py` - the code was right all along.

**File tree**:

```
messenger.py
test_messenger.py
```

**Interface contract**
`greet(name: str) -> str` - returns `"hello, {name}"`.

**Constraints**
- Create `messenger.py` returning `"hello, {name}"`; it never changes afterwards.
- Write the wrong test first, run it, read where the report points.
- Then fix the test and confirm it goes green.
- Command: `python -m pytest test_messenger.py -q`

**Expected** (Driver cell): `1 passed in 0.01s`

In [6]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_messenger.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

1 passed in 0.01s


## E3 - First fixture: shared sample data

**Type:** REAL-WORLD - easy - edits 1 file

**Task**
Three sample prices (`[2.5, 0.0, -12.5]`) keep appearing in tests. Write a
`sample_prices` fixture so the tests request it as a parameter instead of
building the list themselves.

**File tree**:

```
test_prices.py        (you create)
price_utils.py        (provided - do NOT modify)
```

**Interface contract**
- fixture `sample_prices` -> `[2.5, 0.0, -12.5]`
- `format_price(amount: float) -> str` - two decimals.

**Constraints**
- One fixture, three test functions; each takes the fixture as a parameter.
- No `for` loop inside a test - each behavior gets its own test.
- AAA (arrange / act / assert) with blank lines between the parts.
- Command: `python -m pytest test_prices.py -q`

**Expected** (Driver cell): `3 passed in 0.01s`

In [11]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_prices.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

3 passed in 0.01s


## E4 - Shared fixtures to conftest.py + tmp_path

**Type:** REAL-WORLD - medium - writes 2 files, edits 1

**Task**
Tests covering `report_from_file` need real files on disk. Two moves this step:

1. **Move** the `sample_prices` fixture from `test_prices.py` into a new
   `conftest.py` at the lesson root, and delete the local copy. Fixtures in
   `conftest.py` are visible to every test file in that folder.
2. Create `test_files.py` using `tmp_path` - pytest gives you a fresh folder
   per test, so files never leak between tests.

**File tree**:

```
conftest.py           (you create - holds the moved fixture)
test_files.py         (you create)
test_prices.py        (edit: remove the local fixture)
```

**Interface contract**
- `report_from_file(path) -> str` - one `name: signed` line per row, then a
  `net: signed` total line.
- `tmp_path` fixture -> a fresh `Path` folder per test.

**Constraints**
- `test_prices.py` keeps passing after the move (its tests are untouched).
- Test 1: write a file into `tmp_path`, build the report, assert the exact string.
- Test 2: `pytest.raises(FileNotFoundError)` for a missing file.
- Command: `python -m pytest test_prices.py test_files.py -q`

**Expected** (Driver cell): `5 passed in 0.01s`

In [19]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_prices.py", "test_files.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

5 passed in 0.01s


## E5 - Parametrize: one test, many inputs**Type:** REAL-WORLD - medium - writes 1 file**Task**`format_price` has five interesting inputs. Turn them into parametrized caseswith readable `ids`, alongside a parametrized `test_signed` that proves thesame shape for the always-sign formatter.**File tree**:```test_formatting.py    (you create)```**Interface contract**| amount      | format_price | signed   ||-------------|--------------|----------|| 0.0         | "0.00"       | "+0.00"  || 2.5         | "2.50"       | "+2.50"  || -12.5       | "-12.50"     | "-12.50" |**Constraints**- `@pytest.mark.parametrize` with `ids=` that explain each case  (e.g. `"zero"`, `"cents"`, `"negative"`, `"thousands"`, `"sub-cent"`).- Parametrized cases for `format_price`: `0.0` -> `"0.00"`, `2.5` -> `"2.50"`,  `-12.5` -> `"-12.50"`, `1234567.891` -> `"1,234,567.89"`, and `0.004` -> `"0.00"`.- `ids=` exists so a failing case names itself in the report - `[sub-cent]`  beats an anonymous parameter dump. Named cases are the diagnostic, not  decoration.- One parametrized `test_signed` with two cases proving the always-sign rule:  `42.0` -> `"+42.00"`, `-5.5` -> `"-5.50"`.- Command: `python -m pytest test_formatting.py -q`**Expected** (Driver cell): `7 passed in 0.01s`

In [20]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_formatting.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

7 passed in 0.01s


## M1 - Parametrize + tmp_path: report sums

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
The `net:` line of a report is the sum of the amounts. Prove it with
parametrized, file-based tests: different files, same behavior.

**File tree**:

```
test_report_cases.py  (you create)
```

**Interface contract**
`report_from_file(path) -> str` - last line is `net: <signed sum>` followed by a
newline; blank rows are skipped.

**Constraints**
- `@pytest.mark.parametrize` with `ids=`; each case gives the file lines and the
  expected net. The test writes the lines to a `tmp_path` file, builds the
  report, and asserts on its last line.
- At least three cases: mixed income/expense, a single row, a duplicated row.
- One more test: a file with a blank line in the middle still produces the right
  number of report lines.
- Command: `python -m pytest test_report_cases.py -q`

**Expected** (Driver cell): `4 passed in 0.01s`

In [5]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_report_cases.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

4 passed in 0.01s


## M2 - capsys + monkeypatch: capture and swap

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
`print_report` writes lines to the terminal. Testing printed output needs two
fixtures: `capsys` to *capture* the output and `monkeypatch` to *swap* a
function for a test double.

**File tree**:

```
test_print.py         (you create)
```

**Interface contract**
`print_report(rows) -> None` - prints one `name: <signed>` line per row.

**Constraints**
- Test 1: print two rows, read `capsys.readouterr().out`, assert the exact string.
- Test 2: `monkeypatch.setattr("price_utils.signed", ...)` with a lambda that always
  returns `"??.??"`, print one row, assert `"??.??"` appears. This proves
  `print_report` routes through `signed` by name - the seam to swap.
- Command: `python -m pytest test_print.py -q`

**Expected** (Driver cell): `2 passed in 0.01s`

In [15]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_print.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

2 passed in 0.01s


## M3 - pytest.raises: the contract for failure

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
The two loaders have an asymmetric contract: `load_amounts` trusts nothing and
returns `[]` for a missing file; `report_from_file` refuses and raises. Prove
the contract with `pytest.raises`.

**File tree**:

```
test_loaders.py       (you create)
```

**Interface contract**
- `load_amounts(path) -> list[float]` - missing file -> `[]`; blanks and junk
  lines are skipped.
- `report_from_file(path) -> str` - path missing or not a file -> raises
  `FileNotFoundError`.

**Constraints**
- One test per loader success path, plus the two raises cases: a missing file,
  and passing *not a file* (e.g. a directory) as the path.
- Each `pytest.raises` block asserts the exception type only.
- AAA in every test.
- Command: `python -m pytest test_loaders.py -q`

**Expected** (Driver cell): `4 passed in 0.01s`

In [21]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_loaders.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

4 passed in 0.01s


## H1 - Debug drill: fix the broken tests

**Type:** DEBUG - medium - edits 1 file

**Task**
`test_budget.py` is provided **with four planted bugs**. Find and fix all four
so the suite goes green. `budget.py` is correct - do not touch it. Fill in the
bug report table as you go.

**File tree**:

```
test_budget.py        (provided, buggy - you fix it)
budget.py             (provided - do NOT modify)
```

**Hints**
- Bug 1: run the file. The collection error message names exactly what is missing.
- Bug 2: one test passes without proving anything - find the test with no assertion.
- Bug 3: one test asserts the wrong value. Compare the actual numbers.
- Bug 4: one test only fails when the whole file runs. A fixture that is supposed
  to be fresh per test is created once instead. (Think about fixture scope.)

**Bug report** (fill in as you go):

```
# | Bug (what was wrong)          | Fix (what you changed)
--|-------------------------------|--------------------------
1 |                               |
2 |                               |
3 |                               |
4 |                               |
```

**Expected** (Driver cell): `5 passed in 0.01s` - the final state after all four fixes.

```
- Bug (what was wrong)
- Fix (what you changed)

1. NameError: name 'pytest' is not defined 
- add import pytest to the module
2. test_total_spend_returns_float didnt assert
- add assertion result == 0.0
3. false result assertion on test_top_spend_finds_largest
- it should be ("salary", 100.0)
4. fixture set on module level
- set the scope level to function
```

In [13]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_budget.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

5 passed in 0.01s


## H2 - Integration: the full path

**Type:** REAL-WORLD - medium - writes 1 file

**Task**
Tie the pieces together end to end: loader -> sum -> signed total, and a full
byte-exact report.

**File tree**:

```
test_app.py           (you create)
```

**Interface contract**
- `load_amounts(path)` - junk-tolerant loader (M3).
- `report_from_file(path) -> str` - exact bytes ending in `net: <signed>\n`.

**Constraints**
- Test 1: write `12.5`, `-3`, blank, `abc`, `42` into a file; load it, assert the
  list `[12.5, -3.0, 42.0]` and the signed total `"+51.50"`.
- Test 2: write `food,-12.50` / `salary,100.00` / `ride,-8.00`; assert the report
  equals the exact four-line string.
- Happy path only - no `pytest.raises` here.
- Command: `python -m pytest test_app.py -q`

**Expected** (Driver cell): `2 passed in 0.01s`

In [10]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_app.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

2 passed in 0.01s


## Challenge - The shared test suite

**Type:** CHALLENGE - medium - writes 4 files, edits 1

**Task**
Build one folder-wide suite for `price_utils`. Extend `conftest.py` with a
`clean_tx_file` fixture (a ready-made three-row file built from `tmp_path`), then
write four suite files. All files stay **flat in the lesson root** - a `tests/`
package is deliberately NOT used.

**File tree**:

```
conftest.py                    (edit: add the clean_tx_file fixture)
test_suite_formatting.py       (you create)
test_suite_categorize.py       (you create)
test_suite_loaders.py          (you create)
test_suite_report.py           (you create)
```

**Constraints**
- `test_suite_formatting.py`: two parametrized tests (`format_price` with 5 cases,
  `signed` with 2), `ids=` set.
- `test_suite_categorize.py`: one parametrized `categorize` test (income / expense /
  zero, including `-0.0`), plus true/false tests for `is_affordable`.
- `test_suite_loaders.py`: reuse `clean_tx_file` in one test; include a
  `pytest.raises` case.
- `test_suite_report.py`: one `capsys` test plus a byte-exact report test that
  reuses `clean_tx_file`.
- No `@pytest.fixture(autouse=True)` anywhere - every fixture is requested.
- AAA in every test.
- Command: `python -m pytest test_suite_formatting.py test_suite_categorize.py test_suite_loaders.py test_suite_report.py -q`

**Expected** (Driver cell): `20 passed in 0.01s`

In [14]:
import re
import subprocess
import sys

_r = subprocess.run(
    [sys.executable, "-m", "pytest", "test_suite_formatting.py", "test_suite_categorize.py", "test_suite_loaders.py", "test_suite_report.py", "-q"],
    capture_output=True,
    text=True,
)
_out = (_r.stdout + _r.stderr).replace("\r\n", "\n")
_out = re.sub(r" in \d+\.\d+s", " in 0.01s", _out)
print("\n".join(l for l in _out.splitlines() if "[100%]" not in l and l.strip()))

20 passed in 0.01s
